# LTFSR-Meta — Run an experiment on CIFAR-100-LT

This is the **single notebook** you run (locally or on Kaggle). It only sets the
configuration and calls the modules in `src/` — no training logic lives here.

Pipeline: **configure → load data → train one method → evaluate → visualise.**

Choose one of four methods (see `docs/` for the algorithm behind each):
`baseline`, `prototype`, `contrastive`, `meta`.


## 1. Make `src/` importable (works locally and on Kaggle)

In [ ]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    """Return the folder containing `src/` (search cwd, parents, Kaggle inputs)."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if base.exists():
            candidates += [p for p in base.iterdir() if p.is_dir()]
    for path in candidates:
        if (path / "src" / "__init__.py").exists():
            return path
    return Path.cwd()


PROJECT_DIR = find_project_root()
sys.path.insert(0, str(PROJECT_DIR))
print("Project root:", PROJECT_DIR)

## 2. Configuration — the only cell you normally edit

On Kaggle, point `DATA_DIR` at your uploaded dataset (e.g.
`/kaggle/input/cifar-100-lt/CIFAR-100-LT`) and `OUTPUT_DIR` at `/kaggle/working`.


In [ ]:
# --- paths ---
DATA_DIR = PROJECT_DIR / "data" / "CIFAR-100-LT"   # folder with train/ test/ class_counts.json
OUTPUT_DIR = PROJECT_DIR / "outputs"               # where run folders are written
BUILD_DATASET = False   # set True on Kaggle to build CIFAR-100-LT into OUTPUT_DIR first

# --- method: one of "baseline", "prototype", "contrastive", "meta" ---
METHOD = "baseline"

# --- core hyperparameters ---
NUM_CLASSES = 100
BATCH_SIZE = 128
LEARNING_RATE = 0.1     # baseline/prototype/probe use SGD; meta uses Adam (see below)
EPOCHS = 100
SEED = 42
PRETRAINED = True       # ImageNet-pretrained ResNet-18 backbone
NUM_WORKERS = 2
DEVICE = None           # None = auto (CUDA if available), or "cpu" / "cuda"

# --- quick smoke test (None = use the full dataset) ---
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None

# --- contrastive (Method 3) ---
PRETRAIN_EPOCHS = 100
PROBE_EPOCHS = 30
TEMPERATURE = 0.07

# --- few-shot / meta-learning (Method 4) ---
N_WAY = 5
K_SHOT = 5
N_QUERY = 15
EPISODES_PER_EPOCH = 100
META_LR = 0.001

## 3. Imports, reproducibility, device, run folder

In [ ]:
import random

import numpy as np
import torch

from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_split,
                                    make_loader, split_shot_groups)
from src.evaluation import visualize
from src.evaluation.metrics import compute_metrics, format_metrics
from src.trainers.engine import evaluate
from src.utils.experiment import create_run_dir, save_config, save_history, save_metrics
from src.utils.seed import resolve_device, set_seed

set_seed(SEED)
device = resolve_device(DEVICE)
run_dir = create_run_dir(OUTPUT_DIR, run_name=METHOD)
print("Device:", device, "| Run dir:", run_dir)

## 4. (Optional) Build the dataset, then load it

In [ ]:
if BUILD_DATASET:
    import subprocess
    subprocess.run([sys.executable, str(PROJECT_DIR / "data" / "prepare_datasets.py"),
                    "--dataset", "cifar100-lt",
                    "--data_dir", str(OUTPUT_DIR), "--overwrite"], check=True)
    DATA_DIR = OUTPUT_DIR / "CIFAR-100-LT"

assert (DATA_DIR / "train").exists(), f"No train/ folder under {DATA_DIR}"
class_counts = load_class_counts(DATA_DIR)
shot_groups = split_shot_groups(class_counts)
print("Classes per group:", {name: len(ids) for name, ids in shot_groups.items()})
print("Head class count:", max(class_counts.values()), "| Tail class count:", min(class_counts.values()))

In [ ]:
def subsample(dataset, max_samples, seed=SEED):
    """Keep a random subset of an ImageFolder in place (smoke testing only).

    Editing `.samples`/`.targets` (instead of wrapping in Subset) keeps the
    attributes the episodic meta-trainer relies on.
    """
    if max_samples is None or max_samples >= len(dataset.samples):
        return dataset
    rng = random.Random(seed)
    keep = rng.sample(range(len(dataset.samples)), max_samples)
    dataset.samples = [dataset.samples[i] for i in keep]
    dataset.targets = [dataset.targets[i] for i in keep]
    return dataset


# Three views of the data: augmented train (for learning), clean train (for
# prototypes / t-SNE), and the test set (evaluation, the standard LT protocol).
train_aug = subsample(load_split(DATA_DIR, "train", build_transforms(train=True)), MAX_TRAIN_SAMPLES)
train_eval = subsample(load_split(DATA_DIR, "train", build_transforms(train=False)), MAX_TRAIN_SAMPLES)
test_eval = subsample(load_split(DATA_DIR, "test", build_transforms(train=False)), MAX_TEST_SAMPLES)

train_loader = make_loader(train_aug, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
train_eval_loader = make_loader(train_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = make_loader(test_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"train={len(train_aug)} images | test={len(test_eval)} images")

> **Note on evaluation.** CIFAR-100-LT is trained on the long-tailed split and
> evaluated on the *balanced* test set — that is the standard benchmark protocol,
> so we monitor and report on the test set directly (no separate validation
> split). For a stricter setup you could hold out a validation split here.

## 5. Look at the data (long-tail profile)

In [ ]:
visualize.plot_class_frequency(class_counts, run_dir)
visualize.plot_shot_distribution(shot_groups, run_dir)
print("Saved class_frequency.png and shot_distribution.png to", run_dir)

## 6. Train the chosen method

In [ ]:
pretrain_history = None

if METHOD == "baseline":
    from src.trainers.baseline_trainer import train_baseline
    model, history = train_baseline(train_loader, test_loader, NUM_CLASSES, device, run_dir,
                                     epochs=EPOCHS, learning_rate=LEARNING_RATE, pretrained=PRETRAINED)
    result = evaluate(model, test_loader, device, collect_features=True)

elif METHOD == "prototype":
    from src.trainers.prototype_trainer import train_prototype
    model, history = train_prototype(train_loader, test_loader, NUM_CLASSES, device, run_dir,
                                     epochs=EPOCHS, learning_rate=LEARNING_RATE, pretrained=PRETRAINED)
    result = evaluate(model, test_loader, device, collect_features=True)

elif METHOD == "contrastive":
    from src.trainers.contrastive_trainer import train_contrastive
    model, history, pretrain_history = train_contrastive(
        DATA_DIR, train_loader, test_loader, NUM_CLASSES, device, run_dir,
        pretrain_epochs=PRETRAIN_EPOCHS, probe_epochs=PROBE_EPOCHS,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
        pretrain_lr=LEARNING_RATE, probe_lr=LEARNING_RATE,
        temperature=TEMPERATURE, pretrained=PRETRAINED)
    result = evaluate(model, test_loader, device, collect_features=True)

elif METHOD == "meta":
    from src.trainers.meta_trainer import evaluate_meta, train_meta
    encoder, history = train_meta(train_aug, train_eval, device, run_dir,
                                  epochs=EPOCHS, episodes_per_epoch=EPISODES_PER_EPOCH,
                                  n_way=N_WAY, k_shot=K_SHOT, n_query=N_QUERY,
                                  learning_rate=META_LR, pretrained=PRETRAINED, seed=SEED)
    result = evaluate_meta(encoder, train_eval_loader, test_loader, NUM_CLASSES, device)

else:
    raise ValueError(f"Unknown METHOD: {METHOD!r}")

print("Training finished. Test accuracy:", round(result["accuracy"], 4))

## 7. Metrics + save everything for this run

In [ ]:
metrics = compute_metrics(result["y_true"], result["y_pred"], NUM_CLASSES, shot_groups)
print(format_metrics(metrics))

config = {k: v for k, v in dict(
    method=METHOD, num_classes=NUM_CLASSES, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE, epochs=EPOCHS, seed=SEED, pretrained=PRETRAINED,
    device=str(device), data_dir=str(DATA_DIR),
    n_way=N_WAY, k_shot=K_SHOT, n_query=N_QUERY, episodes_per_epoch=EPISODES_PER_EPOCH,
    pretrain_epochs=PRETRAIN_EPOCHS, probe_epochs=PROBE_EPOCHS, temperature=TEMPERATURE,
).items()}

save_config(run_dir, config)
save_metrics(run_dir, metrics)
save_history(run_dir, history)
if pretrain_history is not None:
    pretrain_history.to_csv(run_dir / "pretrain_metrics.csv", index=False)
print("Saved config.json, metrics.json, metrics.csv to", run_dir)

## 8. Visualise the results

In [ ]:
visualize.plot_training_curves(history, run_dir)
visualize.plot_confusion_matrices(result["y_true"], result["y_pred"], NUM_CLASSES, run_dir)
if "features" in result:
    visualize.plot_tsne(result["features"], result["y_true"], run_dir)

print("Figures saved to", run_dir)
print(sorted(p.name for p in run_dir.glob("*.png")))

## 9. Summary

The run folder now contains a complete, reproducible record:

```
outputs/<method>/
├── config.json                      # exact hyperparameters
├── metrics.csv                      # per-epoch history
├── metrics.json                     # final test metrics (incl. many/med/few, g-mean, mcc)
├── best_model.pt                    # best checkpoint
├── curve_loss.png / curve_accuracy.png
├── confusion_matrix.png / confusion_matrix_normalized.png
├── class_frequency.png / shot_distribution.png
└── tsne.png
```

To **compare methods**, change `METHOD` in cell 2 and Run All again — each method
writes to its own folder, so nothing is overwritten. Then compare `g_mean`,
`balanced_accuracy`, and `few_shot_accuracy` across `outputs/*/metrics.json`.
